# Module 6 — Working with MCP Servers

> Part of the **"Develop & Deploy AI Agents on Azure with LangChain, Python and Foundry"** course.

In Module 5 you added **local Python functions** as tools. That works great for a single notebook — but in real life, tools are:

* maintained by **other teams**,
* written in **other languages**,
* shared across **many agents**.

The **Model Context Protocol (MCP)** is the open standard that solves this. This module is your conceptual primer; Module 7 deploys an MCP server to Azure, Module 8 plugs it into the agent.

## 🎯 Learning objectives

By the end of this module you will be able to:

1. Define **MCP** in one sentence and explain why it exists.
2. Name the **three primitives** an MCP server can expose.
3. Compare MCP **transports** (stdio, HTTP+SSE, Streamable HTTP) and pick the right one.
4. Spin up a **tiny local MCP server** in Python with the official SDK.
5. Connect to it from a client and list its tools.

## 🧠 What is MCP?

> **MCP is a JSON-RPC protocol that lets an LLM application discover and call tools, read resources, and use prompts from a separate process.**

It is to AI agents what **USB** is to peripherals: one cable, many devices.

```
  ┌──────────────┐    list_tools     ┌──────────────┐
  │  LLM client  │ ────────────────► │  MCP server  │
  │  (agent)     │ ◄──────────────── │              │
  └──────────────┘    call_tool      └──────────────┘
```

* Same client (your agent) can talk to **any** MCP server.
* Same server can serve **any** MCP client (Claude Desktop, VS Code Copilot, LangChain, Foundry, …).

## 🧩 The three primitives

| Primitive   | Mental model                  | Example                                                |
| ----------- | ----------------------------- | ------------------------------------------------------ |
| **Tool**    | A function the LLM can call.  | `search_web(query)`, `create_jira_ticket(...)`         |
| **Resource**| A blob of context the LLM reads. | A README, a database schema, a log file.             |
| **Prompt**  | A re-usable prompt template.  | `summarize_for_executive(text)`                        |

In this course we focus on **tools** — that's 95 % of MCP usage today.

## 🚚 Transports

| Transport            | When to use                                                | Pros / cons                                |
| -------------------- | ---------------------------------------------------------- | ------------------------------------------ |
| **stdio**            | Client and server on the **same machine**. Easy demos.     | Zero config; no network.                   |
| **HTTP + SSE** (legacy) | Server is remote.                                       | Two endpoints (`/sse`, `/messages`).       |
| **Streamable HTTP**  | Server is remote (recommended in 2025+).                   | One endpoint (`/mcp`); resumable streams.  |

👉 In Module 7 we deploy a **Streamable HTTP** MCP server on Container Apps.

## 🧪 Hands-on: a 10-line MCP server

Let's create the smallest possible MCP server to make the protocol concrete.

In [ ]:
%pip install "mcp[cli]" fastmcp

In [ ]:
from pathlib import Path

Path("mini_mcp_server.py").write_text('''
from fastmcp import FastMCP

mcp = FastMCP("mini-demo")

@mcp.tool
def add(a: int, b: int) -> int:
    """Return the sum of a and b."""
    return a + b

@mcp.tool
def greet(name: str) -> str:
    """Return a friendly greeting."""
    return f"Hello {name}, welcome to MCP!"

if __name__ == "__main__":
    mcp.run()  # stdio transport by default
''')
print("Wrote mini_mcp_server.py")

## 🔌 Connect to the server and list its tools

We spawn the server as a child process (stdio transport) and ask it what it can do.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import sys

params = StdioServerParameters(command=sys.executable, args=["mini_mcp_server.py"])

async with stdio_client(params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        for t in tools.tools:
            print(f"• {t.name}: {t.description}")

        result = await session.call_tool("greet", {"name": "Alice"})
        print("\nCall result:", result.content[0].text)

## ✅ Recap

* MCP is a protocol; **fastmcp** is the easy way to build a server in Python.
* Tools, resources and prompts are the three primitives.
* In a notebook **stdio** is convenient; in production we use **Streamable HTTP**.

Next: **Module 7** packages a real MCP server (web-search) as a container and deploys it to Azure Container Apps.